In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import re
import pandas as pd
import time

options = webdriver.ChromeOptions()
# Uncomment the next line if you wish to run Chrome in headless mode
# options.add_argument('--headless')
driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)

driver.get('https://www.fragrantica.com/search/')

perfumes_data = []

# Function to click the "Show more results" button
def click_show_more():
    try:
        show_more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Show more results')]"))
        )
        show_more_button.click()
        time.sleep(2)  # Adjust based on the site's response time
    except (NoSuchElementException, TimeoutException):
        print("No more results button found or there are no more contents to load.")

# Click the "Show more results" button as many times as needed
# Adjust the range based on how many additional items you want to load
for _ in range(5):  # Example: adjust this number
    click_show_more()

try:
    WebDriverWait(driver, 30).until(
        EC.visibility_of_all_elements_located((By.CSS_SELECTOR, 'div.cell.card.fr-news-box'))
    )
    containers = driver.find_elements(By.CSS_SELECTOR, 'div.cell.card.fr-news-box')[:50]  # Limit for demonstration
    for i, container in enumerate(containers, start=1):
        perfume_name_element = container.find_element(By.CSS_SELECTOR, 'p > a')
        perfume_brand_element = container.find_element(By.CSS_SELECTOR, 'p > small')
        perfume_name = perfume_name_element.text.strip()
        perfume_brand = perfume_brand_element.text.strip()
        link = perfume_name_element.get_attribute('href')
        
        perfume = {
            "Rank": i,
            "Perfume Name": perfume_name,
            "Perfume Brand": perfume_brand,
            "Link": link
        }
        
        driver.get(link)
        WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'div#main-content')))
        
        description_element = driver.find_element(By.XPATH, '//*[@itemprop="description"]/p')
        description = description_element.text
        year_match = re.search(r'\b\d{4}\b', description)
        perfume['Year'] = year_match.group(0) if year_match else 'Unknown'
        
        try:
            rating = driver.find_element(By.CSS_SELECTOR, 'span[itemprop="ratingValue"]').text
            votes = driver.find_element(By.CSS_SELECTOR, 'span[itemprop="ratingCount"]').text
            perfume['Rating'] = rating
            perfume['Votes'] = votes
        except NoSuchElementException:
            perfume['Rating'], perfume['Votes'] = 'Unknown', 'Unknown'

        try:
            perfumer_elements = driver.find_elements(By.XPATH, "//div[contains(@class, 'grid-x grid-padding-x grid-padding-y small-up-2 medium-up-2')]//a")
            perfumers = ', '.join([elem.text for elem in perfumer_elements if elem.text.strip() != ""])
            perfume['Perfumers'] = perfumers
        except NoSuchElementException:
            perfume['Perfumers'] = 'Unknown'

        # Add logic for 'Main Accords' and notes extraction here if necessary
        try:
            accord_elements = driver.find_elements(By.CSS_SELECTOR, 'div.accord-box > div.accord-bar')
            main_accords = {accord_element.text: accord_element.get_attribute('style').split('width: ')[1].rstrip('%;') for accord_element in accord_elements}
        except NoSuchElementException:
            main_accords = {}

        # Scrape the notes
        notes_xpath = {
            'Top Notes': "//h4[.='Top Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Middle Notes': "//h4[.='Middle Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]",
            'Base Notes': "//h4[.='Base Notes']/following-sibling::div//div[contains(@style, 'text-align: center')]/div[2]"
        }
        notes_data = {}
        for note_type, xpath in notes_xpath.items():
            try:
                elements = driver.find_elements(By.XPATH, xpath)
                notes_data[note_type] = [element.text for element in elements if element.text]
            except NoSuchElementException:
                notes_data[note_type] = []

        # Update the perfume dictionary
        perfume.update({
            'Rating': rating,
            'Votes': votes,
            'Perfumers': perfumers,
            'Main Accords': main_accords,
            'Top Notes': notes_data['Top Notes'],
            'Middle Notes': notes_data['Middle Notes'],
            'Base Notes': notes_data['Base Notes']
        })

    except Exception as e:
        print(f"Could not scrape detailed info for {perfume['Perfume Name']}: {e}")
    finally:
        driver.quit()
        
        perfumes_data.append(perfume)
        
        # Navigate back to the search page to continue the loop if necessary
        driver.get('https://www.fragrantica.com/search/')
        
except Exception as e:
    print("An error occurred:", e)
finally:
    driver.quit()

df = pd.DataFrame(perfumes_data)
df.to_csv('top_50_perfumes_extended.csv', index=False)
print("Extended CSV file has been saved.")


SyntaxError: invalid syntax (2047039795.py, line 109)

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time
import re

def get_last_scraped_rank(csv_file):
    try:
        df = pd.read_csv(csv_file)
        return df['Rank'].max()
    except Exception:
        return 0

def append_to_csv(data, csv_file, mode='a'):
    df = pd.DataFrame(data)
    if mode == 'w' or get_last_scraped_rank(csv_file) == 0:
        df.to_csv(csv_file, mode=mode, index=False)
    else:
        df.to_csv(csv_file, mode=mode, index=False, header=False)

options = webdriver.ChromeOptions()
driver = webdriver.Chrome('D:\\downloads\\chromedriver-win64\\chromedriver.exe', options=options)
driver.get('https://www.fragrantica.com/search/')

csv_file = 'top_50_perfumes_extended.csv'
start_rank = get_last_scraped_rank(csv_file)

# Assuming click_show_more() is defined as before
# Update the range based on how many additional items you want to load, considering start_rank

perfumes_data = []
if start_rank == 0:  # Only scrape if starting fresh
    try:
        # Your scraping logic here to fill perfumes_data...
        # Remember to implement click_show_more() logic as needed
        pass  # Replace with actual scraping logic
    finally:
        driver.quit()
    append_to_csv(perfumes_data, csv_file, mode='w')  # Write new file if starting fresh
